# Estudo de Caso 04: Multi-Agent 2v2: Emergência de Team-Play e Auto-Confronto

> **HaxBall RL Suite** - Aprendizado por Reforço Multiagente (MARL), atribuição de crédito coletiva e auto-confronto recursivo (*Self-Play*).

Neste notebook, exploramos:
1. Por que agentes ingênuos sofrem do **Dilema do Futebol de Recreio (*Kindergarten Soccer*)** e como quebrar essa simetria.
2. A engenharia de recompensas do `TeamPlayRewardEngine` (passes, assistências, penalidade de *spacing* e âncora defensiva).
3. O loop de treino recursivo 2v2 (`SelfPlay2v2Trainer`) com aceleração de até 100x.
4. A evolução pelas **4 Fases Cognitivas**: da exploração burra inicial ao team-play avançado.


In [ ]:
import os
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.insert(0, module_path)

from haxball.core.vector import Vec2
from haxball.core.constants import Team
from haxball.core.stadium import Stadium
from haxball.core.game import HaxBallGame
from haxball.rl.rewards.reward_engine import TeamPlayRewardEngine
from haxball.rl.algorithms.self_play.self_play_trainer import SelfPlay2v2Trainer

print('Módulos de MARL e Team-Play carregados com sucesso!')


## 1. O TeamPlayRewardEngine e a Atribuição de Crédito

A recompensa de cada jogador combina componentes compartilhadas e individuais:
* **Passe Concluído:** $+2.5$ para o passador, $+1.5$ para o receptor em velocidade.
* **Assistência Oficial:** $+4.0$ adicionais se o passe resultar em gol.
* **Penalidade de Spacing:** $-0.03 \times \text{overlap}$ quando dois companheiros estão a menos de 75 px.
* **Cobertura Defensiva:** $+0.02$/tick para o jogador mais recuado entre a bola e a própria meta.


In [ ]:
map_path = os.path.join('..', 'haxball', 'maps', 'futsal_2v2.hbs')
stad = Stadium.load_from_file(map_path)
game = HaxBallGame(stadium=stad, red_players_count=2, blue_players_count=2)
reward_engine = TeamPlayRewardEngine()
reward_engine.reset(game)

r1, r2 = [p for p in game.players if p.team == Team.RED]

# Cenário A: Jogadores aglomerados (sem espaçamento)
r1.pos = Vec2(0.0, 0.0)
r2.pos = Vec2(15.0, 0.0)  # Apenas 15 px de distância (< 75 px)
game.ball.pos = Vec2(40.0, 0.0)
rewards_crowd = reward_engine.compute_team_rewards(game, game.step({}))

# Cenário B: Jogadores bem espaçados (um na pressão, outro na cobertura)
r1.pos = Vec2(20.0, 0.0)   # Pressionando a bola
r2.pos = Vec2(-100.0, 50.0) # Cobrindo a retaguarda
rewards_spaced = reward_engine.compute_team_rewards(game, game.step({}))

print(f'Cenário Aglomerado (Kindergarten Soccer) - Reward R1: {rewards_crowd[r1.player_id]:+.3f} | R2: {rewards_crowd[r2.player_id]:+.3f}')
print(f'Cenário Espaçado e Tático                 - Reward R1: {rewards_spaced[r1.player_id]:+.3f} | R2: {rewards_spaced[r2.player_id]:+.3f}')
print('✓ A penalidade de spacing quebra a aglomeração indesejada!')


## 2. Treinamento Recursivo 2v2 (Self-Play)

Executamos 1.500 passos rápidos de simulação multiagente com Parameter Sharing.


In [ ]:
trainer = SelfPlay2v2Trainer(game=game, rollout_steps=256)

steps_log = []
reward_log = []
passes_log = []

print('Executando rollouts de Self-Play...')
for cycle in range(1, 11):
    stats = trainer.step_multistep(150)
    steps_log.append(stats['total_steps'])
    reward_log.append(stats['mean_reward'])
    passes_log.append(stats['passes'])
    print(f'Ciclo {cycle:2d} | Passos: {stats["total_steps"]:5d} | Reward Médio: {stats["mean_reward"]:+.2f} | Passes: {stats["passes"]}')

# Visualização gráfica da telemetria
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

ax1.plot(steps_log, reward_log, 'o-', color='#4CAF50', linewidth=2.5)
ax1.set_title('Evolução do Reward Coletivo em Self-Play', fontsize=12, fontweight='bold')
ax1.set_xlabel('Passos de Simulação')
ax1.set_ylabel('Reward Médio')
ax1.grid(True, alpha=0.3)

ax2.plot(steps_log, passes_log, 's-', color='#2196F3', linewidth=2.5)
ax2.set_title('Frequência de Passes e Tabelas Coletivas', fontsize=12, fontweight='bold')
ax2.set_xlabel('Passos de Simulação')
ax2.set_ylabel('Total de Passes Concluídos')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## 3. As 4 Fases Cognitivas do Aprendizado

1. 🔴 **Fase 1: Exploração Burra ($< 5.000$ passos):** Ações puramente aleatórias, giros descontrolados e gols contra acidentais.
2. 🟠 **Fase 2: Perseguição da Bola ($5.000$ a $25.000$ passos):** Os agentes aprendem o gradiente de aproximação e correm na direção da bola.
3. 🟡 **Fase 3: Alinhamento ao Gol e Espaçamento ($25.000$ a $70.000$ passos):** Surgem chutes direcionados às traves e separação mútua dos colegas de time.
4. 🟢 **Fase 4: Team-Play e Passes Coordenados ($> 70.000$ passos):** Troca consciente de passes quando pressionado, assistências e cobertura defensiva.
